# kluster.ai

[kluster.ai](https://kluster.ai) is a inference service that provides access to a variety of high-performance LLMs including Meta's Llama 3.1 and Llama 3.3 models, DeepSeek R1 and V3, Gemma 3 27B and more. You can find all of [kluster.ai's available models](https://docs.kluster.ai/get-started/models/) in the documentation.

kluster.ai provides an API through which developers can perform [real-time](https://docs.kluster.ai/get-started/start-building/real-time/) and [batch](https://docs.kluster.ai/get-started/start-building/batch/) inference, [fine-tune]() their models, and more. For details of all kluster.ai API features, head to the [API reference](https://docs.kluster.ai/api-reference/reference/).

This notebook goes over how to use LangChain with kluster.ai for chat models via the `chat.invoke` endpoint.

## Prerequisites

Before getting started, ensure you have the following:

- **A kluster.ai account** - sign up on the <a href="https://platform.kluster.ai/signup" target="_blank">kluster.ai platform</a> if you don't have one
- **A kluster.ai API key** - after signing in, go to the <a href="https://platform.kluster.ai/apikeys" target="_blank">**API Keys**</a> section and create a new key. For detailed instructions, check out the <a href="/get-started/get-api-key/" target="_blank">Get an API key</a> guide
- **LangChain community and core installed** - you can install them by running:


In [1]:
%pip install --upgrade --quiet langchain-core langchain-community

## Setup

In this notebook, we'll use Python's `getpass` module to safely input the key. After execution, please provide your unique kluster.ai API key (ensure no spaces).

In [2]:
# Get a new token from https://platform.kluster.ai if not already set
import os
from getpass import getpass

# Only prompt for API token if not already set in environment
if "KLUSTERAI_API_KEY" not in os.environ:
    print("Please enter your kluster.ai API token:")
    KLUSTERAI_API_TOKEN = getpass()
    os.environ["KLUSTERAI_API_KEY"] = KLUSTERAI_API_TOKEN
else:
    print("Using existing KLUSTERAI_API_KEY from environment")

Please enter your kluster.ai API token:


 ········


With the API key already set, import `ChatKlusterAI` from LangChain community chat models, and initialize it by passing the model. This example uses `klusterai/Meta-Llama-3.1-8B-Instruct-Turbo`, but feel free to try any other of the [supported models](https://docs.kluster.ai/get-started/models/).

In [3]:
from langchain_community.chat_models import ChatKlusterAI

# Initialize chat with model
chat = ChatKlusterAI(model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo")

You will use this `chat` instance throughout this notebook.

## Invocation

Once you've instantiated the `chat` client, you can easily create a real-time inference with the `invoke` method.


In [4]:
# Set request
messages = [
    (
        "system",
        "You are a professional translator for English to French. Translate directly without explanations.",
    ),
    ("human", "I love programming."),
]

# Invoke
chat.invoke(messages)

AIMessage(content="J'adore le programmation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 54, 'total_tokens': 63}, 'model_name': 'klusterai/Meta-Llama-3.1-8B-Instruct-Turbo', 'finish_reason': 'stop'}, id='run-28374235-6dcc-4aa9-867a-892311160a31-0')

### Using `HumanMessage` and `SystemMessage`

You can use `HumanMessage` and `SystemMessage` to quickly format your messages and provide context and instructions to the model.

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage

# Set request with system message
messages = [
    SystemMessage(
        content="You are a professional translator for English to French. Translate directly without explanations."
    ),
    HumanMessage(content="I love programming."),
]

# Invoke
chat.invoke(messages)

AIMessage(content="J'adore le programmation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 54, 'total_tokens': 63}, 'model_name': 'klusterai/Meta-Llama-3.1-8B-Instruct-Turbo', 'finish_reason': 'stop'}, id='run-5dafb55e-7859-4e11-8792-4d3548f539e9-0')

### Using different models

kluster.ai offers many models that you can use with LangChain. You can find all of [kluster.ai's available models](https://docs.kluster.ai/get-started/models/) in the documentation.

To use a different model, replace the model API name in the `model` parameter.

In [6]:
# Using the larger 405B parameter model
large_model_chat = ChatKlusterAI(
    model="klusterai/Meta-Llama-3.1-405B-Instruct-Turbo",
)

# Different model invoke
large_model_chat.invoke(messages)

AIMessage(content="J'adore la programmation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 54, 'total_tokens': 63}, 'model_name': 'klusterai/Meta-Llama-3.1-405B-Instruct-Turbo', 'finish_reason': 'stop'}, id='run-b0533f2a-d494-4187-8dad-4b2cae73e690-0')

### Controlling model parameters

In addition to providing a `model`, you can control various parameters such as `temperature` to adjust the model's creativity level. For more parameters, check kluster.ai API [chat completion reference](https://docs.kluster.ai/api-reference/reference/#create-chat-completion).

In [7]:
# Using a more deterministic output with lower temperature
precise_chat = ChatKlusterAI(
    model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo", temperature=0.1
)

# Invoke
precise_chat.invoke(messages)

AIMessage(content="J'adore le programmation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 54, 'total_tokens': 63}, 'model_name': 'klusterai/Meta-Llama-3.1-8B-Instruct-Turbo', 'finish_reason': 'stop'}, id='run-4b9cfb36-d0e5-4320-8dfa-ec721e01704f-0')

## Async and streaming functionality

`ChatKlusterAI` also supports both async calls and streaming functionalities. For async, you can use the `agenerate` endpoint. 

In [8]:
from langchain_core.messages import HumanMessage
from langchain_core.callbacks import StreamingStdOutCallbackHandler

# Example of async generation
messages = [
    HumanMessage(
        content="Translate this sentence from English to French. I love programming."
    )
]

# Async invoke
await chat.agenerate([messages])

LLMResult(generations=[[ChatGeneration(text='The translation of the sentence "I love programming" to French is:\n\n"Jeprus de programmer"\n\nHowever the better way to translate would be having that the resume of the meaning stays intact when is translated to french. \n\nthe full proper translation of that sentence to french is: \n\n"J\'adore programmer"', generation_info={'finish_reason': 'stop'}, message=AIMessage(content='The translation of the sentence "I love programming" to French is:\n\n"Jeprus de programmer"\n\nHowever the better way to translate would be having that the resume of the meaning stays intact when is translated to french. \n\nthe full proper translation of that sentence to french is: \n\n"J\'adore programmer"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 47, 'total_tokens': 111}, 'model_name': 'klusterai/Meta-Llama-3.1-8B-Instruct-Turbo', 'finish_reason': 'stop'}, id='run-56b88132-dd23-454b-a5ea-1e4fe552d3d6-0')

For streaming, you need to set `streaming` to true.

In [9]:
# Example of streaming functionality
streaming_chat = ChatKlusterAI(
    model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo",
    streaming=True,
    verbose=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# Stream invoke
streaming_chat.invoke(messages)

The translation of the sentence is :

"J'adore programmer."

AIMessage(content='The translation of the sentence is :\n\n"J\'adore programmer."', additional_kwargs={}, response_metadata={}, id='run-9b289839-9900-4cf2-b51c-aaaee098ddfa-0')

## Tool calling

kluster.ai supports tool calling functionality with models like `Meta-Llama-3.1-405B` and `Meta-Llama-3.3-70B`. This lets you define tools that the model can call to perform specific actions. For a complete list of models that support tool calling, please refer to the [kluster.ai models comparison table](https://docs.kluster.ai/get-started/models/#model-comparison-table).

First, define the tools. Note that for this snippet you'll need `pydantic`.

In [10]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

# Define a simple tool using the @tool decorator
class GetWeather(BaseModel):
    """Get the current weather in a given location"""

    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")


# Define a tool using Pydantic for more complex parameters
class SearchQuery(BaseModel):
    """Search for information on a given topic."""

    query: str = Field(..., description="The search query")
    max_results: int = Field(5, description="Maximum number of results to return")

With the tools defined, you need to initialize the `chat` with a [model that supports tools](https://docs.kluster.ai/get-started/models/#model-comparison-table). Then, you can bind the tools to the model, create and send the message.

In [11]:
# Set up the chat model with tool capabilities
tool_capable_chat = ChatKlusterAI(model="klusterai/Meta-Llama-3.1-405B-Instruct-Turbo")

# Bind the tools to the model
llm_with_tools = tool_capable_chat.bind_tools([GetWeather])

# Create a message that will trigger tool usage
weather_message = "What's the weather in San Francisco?"
print(f"Prompt: {weather_message}")

# Send the query to the model
response = llm_with_tools.invoke(weather_message)
response

Prompt: What's the weather in San Francisco?


AIMessage(content='', additional_kwargs={'tool_calls': [ChatCompletionMessageToolCall(id='chatcmpl-tool-1fd60756af0849ed8d67e7766bcdf3ca', function=Function(arguments='{"location": "San Francisco, CA"}', name='GetWeather'), type='function')]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 263, 'total_tokens': 284}, 'model_name': 'klusterai/Meta-Llama-3.1-405B-Instruct-Turbo', 'finish_reason': 'tool_calls'}, id='run-e635d4aa-2297-4574-a22f-a9f9c116393d-0', tool_calls=[{'name': 'GetWeather', 'args': {'location': 'San Francisco, CA'}, 'id': 'chatcmpl-tool-1fd60756af0849ed8d67e7766bcdf3ca', 'type': 'tool_call'}])

Lastly, verify the tool call from the response.

In [12]:
# Examine the tool calls
print(f"Tool calls: {response.tool_calls}\n")

# Example of processing the tool call
print("Example of processing tool calls and responding:")
if response.tool_calls:
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_weather":
            location = tool_call["args"]["location"]
            weather_result = get_weather(location)
            print(weather_result)

Tool calls: [{'name': 'GetWeather', 'args': {'location': 'San Francisco, CA'}, 'id': 'chatcmpl-tool-1fd60756af0849ed8d67e7766bcdf3ca', 'type': 'tool_call'}]

Example of processing tool calls and responding:


## Using `PromptTemplate` with `ChatKlusterAI`

You can combine `ChatKlusterAI` with LangChain's `PromptTemplate` for more structured interactions. The following example creates a template for a translation inference call, using a `source_language`, `target_language` and a `text_to_translate`. It also chains the prompt with the chat model with the pipe operator.

Note that with `PromptTemplate` you need to use `HumanMessagePromptTemplate` and `SystemMessagePromptTemplate` for message formatting, or just provide the message directly formatted.

In [13]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

# Define prompt messages with placeholders
messages = [
    SystemMessagePromptTemplate.from_template(
        "You are a professional translator from {source_language} to {target_language}."
    ),
    HumanMessagePromptTemplate.from_template("{text_to_translate}"),
]

# Create a chat prompt template
prompt_template = ChatPromptTemplate.from_messages(messages)

# Format the prompt with input values
formatted_messages = prompt_template.format_messages(
    source_language="English",
    target_language="French",
    text_to_translate="I like programming in Python."
)

# Now you can pass `formatted_messages` to a chat model manually
response = chat.invoke(formatted_messages)

print(f"Response: {response.content}")

Response: Le Python est un langage de programmation très utile. 

(A French translation of your statement: "I like programming in Python".)

Son vervabilité et sa lisibilité, rendent ses applications possibles dans de nombreux domaines tels que l'intelligence artificielle, les algorithmes, les jeux vidéo et bien d'autres.

(Type "Pouvez-vous traduire cela?" if you would like to have more phrases translated.)


### Create a runnable sequence

You can also pipe the template directly into the chat model with the pipe operators to create a `RunnableSequence`.

In [14]:
# Pipe the template directly into the chat model
translation_chain = prompt_template | chat

# Run the chain
response = translation_chain.invoke(
    {
        "source_language": "English",
        "target_language": "French",
        "text_to_translate": "I like programming in Python.",
    }
)

print(f"Response: {response.content}")

Response: Vous aimez le langage de programmation Python. C'est un excellent choix, Python est connu pour sa simplicité et sa facilité d'utilisation, convient très bien pour les débutants mais également pour les programmes complexes.

Pouvez-vous me dire quel type de programmation vous réalisez avec Python ? 

Par exemple : applications, jeux, traitement de données, analyse de texte...


## Batch Processing with kluster.ai

For handling larger workloads, kluster.ai supports [batch processing](https://docs.kluster.ai/get-started/start-building/batch/). To do so, you can create an array of requests and pass them as a `HumanMessage`. In this example, a `SystemMessage` is also included.

In [15]:
from langchain_core.messages import HumanMessage, SystemMessage

# Example of processing multiple translation requests
texts_to_translate = [
    "I love programming.",
    "The weather is nice today.",
    "My dog ate my homework.",
]

print("Processing batch of translation requests...\n")

# Process each request
for text in texts_to_translate:
    # Set request
    system_msg = SystemMessage(
        content="You are a professional French translator. Translate directly without explanations."
    )
    human_msg = HumanMessage(content=text)

    # Invoke batch
    response = chat.invoke([system_msg, human_msg])

    print(f"Input: {text}")
    print(f"Translation: {response.content}\n")

Processing batch of translation requests...

Input: I love programming.
Translation: J'adore le langage de programmation.

Input: The weather is nice today.
Translation: Le temps est agréable aujourd'hui.

Input: My dog ate my homework.
Translation: Mon chien a mangé mon devoir.



## Error Handling

When working with any API, proper error handling is important. To do so, you can define a `try` and `except` flow to retry the call after a given timeout, and error out if you've achieved a certain number of attemps.

In [16]:
import time

print("Example of error handling:")

# Function to retry translation if error
def safe_translation(text, max_retries=3):
    """Safely translate text with retries"""
    retry_count = 0

    # Retry again if allowed
    while retry_count < max_retries:
        try:
            # Set request
            system_msg = SystemMessage(
                content="You are a professional French translator. Translate directly."
            )
            human_msg = HumanMessage(content="I love programming.")

            # Invoke
            response = chat.invoke([system_msg, human_msg])
            return f"Translation successful: {response.content}"

        except Exception as e:
            # Error handling
            retry_count += 1
            if retry_count >= max_retries:
                return f"Failed after {max_retries} attempts. Error: {str(e)}"
            print(f"Attempt {retry_count} failed. Retrying in 2 seconds...")

            # Try after 2 seconds
            time.sleep(2)


# Try to translate with error handling
result = safe_translation("I enjoy learning new technologies.")
print(result)

Example of error handling:
Translation successful: J'adore faire du programmation.
